In [1]:
cd ..

/home/karaman/Documents/bet


In [2]:
from src.utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
from pyspark.sql.functions import pandas_udf
import src.utils.config as config
from src.ingestion.churn_label_generator import generate_churn_labels



In [3]:
spark = get_spark()
spark.catalog.clearCache()
config_ = config.DataGenConfig()
players_silver = spark.read.parquet("./data/silver/players")
sessions_silver = spark.read.parquet("./data/silver/sessions")
transactions_silver = spark.read.parquet("./data/silver/transactions")
silver_money_events = spark.read.parquet("./data/silver/money_events")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/21 01:52:58 WARN Utils: Your hostname, karaman-VMware-Virtual-Platform, resolves to a loopback address: 127.0.1.1; using 192.168.182.129 instead (on interface ens33)
26/01/21 01:52:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/21 01:52:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:

players_silver = players_silver.drop('player_id')
transactions_silver = transactions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
sessions_silver = sessions_silver.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')
silver_money_events = silver_money_events.withColumn( "player_idx",
    F.regexp_replace("player_id", "[^0-9]", "").cast("long")).drop('player_id')


In [5]:
silver_money_events = silver_money_events.withColumn('days_diff', F.date_diff(F.lit(config_.end_date), F.col('event_ts')))
sessions_silver = sessions_silver.withColumn('days_diff', F.date_diff(F.lit(config_.end_date), F.col('session_date')))
transactions_silver = transactions_silver.withColumn('days_diff', F.date_diff(F.lit(config_.end_date),F.to_date(F.col('transaction_ts'))))

In [6]:
df_churn = generate_churn_labels(sessions_silver, config_)
df_churn = (
    df_churn
    .join(
        players_silver.select("player_idx"),
        on="player_idx",
        how="inner"
    )
)
#df_churn.write.mode("append").parquet("./data/bronze/churn_labels")


In [7]:
df_churn.show(4)

+----------+------------------+-----------------+--------+--------------+
|player_idx|first_session_date|last_session_date|churn_7d|reference_date|
+----------+------------------+-----------------+--------+--------------+
|     96204|        2024-02-10|       2024-06-30|   false|    2024-06-30|
|     50219|        2024-03-13|       2024-06-29|   false|    2024-06-30|
|      3091|        2024-03-14|       2024-06-29|   false|    2024-06-30|
|     37282|        2024-03-08|       2024-06-29|   false|    2024-06-30|
+----------+------------------+-----------------+--------+--------------+
only showing top 4 rows


In [8]:
player_snapshot = (players_silver
                   .select('player_idx','country','age_bucket','device_type','acquisition_channel', 'registration_date', 'risk_segment', 'lifecycle_stage', F.col('current_balance').alias('current_balance'))
                   .join(df_churn.select('player_idx','first_session_date','last_session_date'),
                         on='player_idx',
                         how='left')
                         .withColumn('days_since_last_session', 
                                     F.date_diff(F.lit(config_.end_date), F.col('last_session_date')))
                         .drop('reference_date')

)
player_snapshot.show()

+----------+-------+----------+-----------+-------------------+-----------------+------------+---------------+------------------+------------------+-----------------+-----------------------+
|player_idx|country|age_bucket|device_type|acquisition_channel|registration_date|risk_segment|lifecycle_stage|   current_balance|first_session_date|last_session_date|days_since_last_session|
+----------+-------+----------+-----------+-------------------+-----------------+------------+---------------+------------------+------------------+-----------------+-----------------------+
|         0|     EN|     25-34|    desktop|            organic|       2024-03-15|     unknown|            new|            123.84|              NULL|             NULL|                   NULL|
|      1002|     EN|     35-44|    desktop|               paid|       2024-01-09|     unknown|            new|             45.05|              NULL|             NULL|                   NULL|
|     10000|     EN|     35-44|     mobile|  

## Window functions for rolling computations 7days or 30days ago

In [ ]:
'''
sessions_silver_sec = (sessions_silver
                   .withColumn('session_date_days', 
                F.datediff(F.col("session_date"), F.lit("1970-01-01"))
)
)

window_7d = (Window.partitionBy('player_id').orderBy('session_date_days').rangeBetween(-7,0))
window_30d = (Window.partitionBy('player_id').orderBy('session_date_days').rangeBetween(-30,0))
sessions_silver_sec = (sessions_silver_sec
            .withColumn('num_sessions_7d', F.count('session_id').over(window_7d))
            .withColumn('num_sessions_30d', F.count('session_id').over(window_30d))
            .withColumn('avg_sessions_duration_30d', F.avg('session_duration_sec').over(window_30d))
)

((sessions_silver_sec.groupBy('player_id')
 .agg(
     F.max('session_date_days').alias('session_date_days'))
)
.join(sessions_silver_sec.select('player_id', 'session_date_days','session_date','num_sessions_7d','num_sessions_30d','avg_sessions_duration_30d'), on=['player_id', 'session_date_days'], how='inner')
.show()
)

'''

# player_behavior

### player_behavior based on sessions activity

In [9]:
player_behavior = (sessions_silver.withColumn('days_diff', F.date_diff(F.lit(config_.end_date), F.col('session_date')))
 .filter(F.col('days_diff') <= 30)
 .groupBy('player_idx')
 .agg(
            F.sum(F.when(F.col('days_diff') <= 7, 1).otherwise(0)).alias('sessions_7d'),
            F.sum(F.when(F.col('days_diff') <= 7, F.col('signed_amount')).otherwise(0)).alias('net_game_result_7d'),

            F.count('*').alias('sesions_30d'),
            F.avg(F.col('session_duration_sec')).cast('int').alias('avg_session_duration_sec_30d'),
            F.avg(F.col('total_bet_amount')).alias('avg_bet_amount_30d'),
            F.sum(F.col('signed_amount')).alias('net_game_result_30d'),
 )
                
)
player_behavior.show()

+----------+-----------+------------------+-----------+----------------------------+------------------+-------------------+
|player_idx|sessions_7d|net_game_result_7d|sesions_30d|avg_session_duration_sec_30d|avg_bet_amount_30d|net_game_result_30d|
+----------+-----------+------------------+-----------+----------------------------+------------------+-------------------+
|      3091|          6|442.94999999999993|         24|                        1866|          50.38125| 1610.7599999999998|
|     71648|         10|            278.79|         28|                        1957| 53.59142857142858|            1496.51|
|     86791|          4|             114.2|         24|                        2011|52.742916666666666|            1355.48|
|     32667|          0|               0.0|         10|                        1593| 55.67999999999999|             666.19|
|     15663|          0|               0.0|          2|                        1565| 79.28999999999999|             142.43|
|     36

In [10]:
player_snapshot.show()

+----------+-------+----------+-----------+-------------------+-----------------+------------+---------------+------------------+------------------+-----------------+-----------------------+
|player_idx|country|age_bucket|device_type|acquisition_channel|registration_date|risk_segment|lifecycle_stage|   current_balance|first_session_date|last_session_date|days_since_last_session|
+----------+-------+----------+-----------+-------------------+-----------------+------------+---------------+------------------+------------------+-----------------+-----------------------+
|         0|     EN|     25-34|    desktop|            organic|       2024-03-15|     unknown|            new|            123.84|              NULL|             NULL|                   NULL|
|      1002|     EN|     35-44|    desktop|               paid|       2024-01-09|     unknown|            new|             45.05|              NULL|             NULL|                   NULL|
|     10000|     EN|     35-44|     mobile|  

### player behavior based on any money event (financial + bet)

In [11]:

### Correct ###
silver_money_events_net = (silver_money_events
 .filter(F.col('days_diff') <= 30)
 .groupBy('player_idx')
 .agg(
            F.sum(F.when(F.col('days_diff') <= 7, F.col('signed_amount')).otherwise(0)).alias('net_amount_result_7d'),
            F.sum(F.col('signed_amount')).alias('net_amount_result_30d'),
 )        
)

silver_money_events_net.show()

+----------+--------------------+---------------------+
|player_idx|net_amount_result_7d|net_amount_result_30d|
+----------+--------------------+---------------------+
|     10959|                 0.0|   1199.3000000000002|
|     13723|   607.4799999999999|   2186.5299999999993|
|     15371|                 0.0|               175.85|
|     17048|               71.21|   215.22999999999996|
|     23492|  369.78000000000003|              1047.22|
|      2453|  389.96999999999997|              1287.34|
|     26543|  290.15000000000003|              2084.23|
|     30421|              978.52|              2756.04|
|     32912|                 0.0|                65.87|
|     35484|              161.94|   1058.8600000000001|
|     37282|              321.61|              1636.52|
|     40634|              877.72|   1563.4499999999998|
|     44901|   590.5699999999999|              1395.88|
|     45298|              169.96|               251.93|
|      4894|              221.82|              1

In [12]:
w = Window.partitionBy("player_idx").orderBy(F.col("event_ts").desc())
player_30d = (silver_money_events
 .filter(F.col('days_diff') >=30)
 .withColumn('rn', F.row_number().over(w))
.filter(F.col('rn') ==1)
)
player_30d_act= (players_silver
                   .select('player_idx','current_balance')
                   .join(player_30d.select('player_idx','balance_after_txn'),
                         on='player_idx',
                         how='left')
                             .withColumn(
                              "balance_30d_ago",
                              F.coalesce("balance_after_txn", "current_balance"))
                              .drop( 'current_balance', 'balance_after_txn')
)
player_30d_act.show()

+----------+------------------+
|player_idx|   balance_30d_ago|
+----------+------------------+
|     10000|2194.8799999999997|
|     10037|             73.36|
|     10030|            188.47|
|     10011|            101.87|
|     10010| 3800.259999999999|
|     10068|1728.4800000000002|
|     10038|            117.26|
|     10046|            188.65|
|     10065|           4922.81|
|       100|             82.05|
|      1007|            1826.4|
|     10024|            4901.9|
|     10006|            107.31|
|     10048| 2464.350000000001|
|     10018|3864.3099999999995|
|     10005|2381.5299999999997|
|     10056|2176.8299999999995|
|     10012|               7.7|
|     10015|             185.8|
|      1006|            580.09|
+----------+------------------+
only showing top 20 rows


In [13]:
w = Window.partitionBy("player_idx").orderBy(F.col("event_ts").desc())
player_7d = (silver_money_events
 .filter(F.col('days_diff') >=7)
 .withColumn('rn', F.row_number().over(w))
.filter(F.col('rn') ==1)
)
player_7d_act= (players_silver
                   .select('player_idx','current_balance')
                   .join(player_7d.select('player_idx',F.col('balance_after_txn')),
                         on='player_idx',
                         how='left')
                             .withColumn(
                              "balance_7d_ago",
                              F.coalesce("balance_after_txn", "current_balance"))
                              .drop('current_balance', 'balance_after_txn')
)
player_7d_act.show()

+----------+------------------+
|player_idx|    balance_7d_ago|
+----------+------------------+
|     10000|3542.7700000000013|
|     10037|             73.36|
|     10030|            188.47|
|     10011|            101.87|
|     10010| 5353.959999999998|
|     10068|2941.2800000000007|
|     10038|            117.26|
|     10046|            188.65|
|     10065|           5843.64|
|       100|             82.05|
|      1007| 3356.579999999999|
|     10024| 5999.590000000001|
|     10006|            107.31|
|     10048| 3589.680000000002|
|     10018| 5700.889999999999|
|     10005|2862.4700000000003|
|     10056|           2726.75|
|     10012|               7.7|
|     10015|             185.8|
|      1006| 709.0200000000001|
+----------+------------------+
only showing top 20 rows


In [14]:
transaction_behavior = (transactions_silver
.filter(F.col('days_diff')<=30)
.groupBy(F.col('player_idx'))
.agg(
    F.sum(F.when((F.col('transaction_type')=='withdrawal') & (F.col('success_flag')==False),1).otherwise(0)).alias('failed_withdrawals_30d'),
    F.sum(F.when(F.col('transaction_type')=='deposit',1).otherwise(0)).alias('deposit_count_30d'),
    F.sum(F.when(F.col('transaction_type')=='withdrawal',1).otherwise(0)).alias('withdrawal_count_30d'),
)
.withColumn(        'withdrawal_ratio',
    F.when(
            F.col("deposit_count_30d") > 0,
            F.col("withdrawal_count_30d") / F.col("deposit_count_30d")
        ).otherwise(F.lit(0.0))
)
)
transaction_behavior.show()

+----------+----------------------+-----------------+--------------------+----------------+
|player_idx|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|
+----------+----------------------+-----------------+--------------------+----------------+
|      2040|                     0|                1|                   0|             0.0|
|     21342|                     0|                1|                   0|             0.0|
|     22165|                     0|                0|                   1|             0.0|
|      2250|                     0|                1|                   0|             0.0|
|     39713|                     0|                1|                   0|             0.0|
|     40132|                     0|                1|                   0|             0.0|
|     46044|                     0|                1|                   0|             0.0|
|     56553|                     0|                0|                   1|      

In [15]:
gold_player_behavior = (players_silver.select('player_idx')
                        .join(silver_money_events_net, how='left', on='player_idx') 
                        .join(transaction_behavior, how='left', on='player_idx') 
                        .join(player_behavior, how='left', on='player_idx') 
                        .join(player_7d_act, how='left', on='player_idx') 
                        .join(player_30d_act, how='left', on='player_idx') 

)
gold_player_behavior.show()

+----------+--------------------+---------------------+----------------------+-----------------+--------------------+----------------+-----------+------------------+-----------+----------------------------+------------------+-------------------+------------------+------------------+
|player_idx|net_amount_result_7d|net_amount_result_30d|failed_withdrawals_30d|deposit_count_30d|withdrawal_count_30d|withdrawal_ratio|sessions_7d|net_game_result_7d|sesions_30d|avg_session_duration_sec_30d|avg_bet_amount_30d|net_game_result_30d|    balance_7d_ago|   balance_30d_ago|
+----------+--------------------+---------------------+----------------------+-----------------+--------------------+----------------+-----------+------------------+-----------+----------------------------+------------------+-------------------+------------------+------------------+
|         0|                NULL|                 NULL|                  NULL|             NULL|                NULL|            NULL|       NULL|  

In [16]:
sessions_silver.show(2)

+--------------------+-------+--------------------+---------+----------------+----------------+-----------+------------+-------------+------------------+----------+---------+
|          session_id|game_id|session_duration_sec|bet_count|total_bet_amount|total_win_amount|device_type|session_date|signed_amount| balance_after_txn|player_idx|days_diff|
+--------------------+-------+--------------------+---------+----------------+----------------+-----------+------------+-------------+------------------+----------+---------+
|000010d3-854e-4bd...|   G122|                2919|        7|           88.01|           11.78|    desktop|  2024-06-23|        11.78|3190.4900000000016|     53660|        7|
|00003c24-fd0b-4ca...|    G17|                   4|       16|           27.51|          104.38|    desktop|  2024-05-07|       104.38| 7020.280000000001|     32087|       54|
+--------------------+-------+--------------------+---------+----------------+----------------+-----------+------------+-----

In [17]:
date_range = spark.sql(
    f"SELECT explode(sequence(to_date('{config_.start_date}'), to_date('{config_.end_date}'), interval 1 day)) AS event_date"
)

df_pl_date = (player_snapshot
              .select('player_idx', 'registration_date','first_session_date', 'last_session_date').crossJoin(date_range)
)
df_num_of_sessions = (sessions_silver
.groupBy('player_idx', 'session_date')
.agg(F.count('*').alias('num_of_session')))
#.orderBy('player_idx', 'session_date' ))

df_sessions = (df_pl_date
               .filter(F.col('first_session_date').isNotNull())
.join(df_num_of_sessions
      .select('num_of_session', F.col('session_date').alias('event_date'), 'player_idx'),
        how='left', on=['player_idx','event_date'])
)


In [20]:
df_sessions_sec = (df_sessions
                   .withColumn('session_date_days', 
                F.datediff(F.col("event_date"), F.lit("1970-01-01"))
)
)

window_7d = (Window.partitionBy('player_idx').orderBy('session_date_days').rangeBetween(-6,0))
window_30d = (Window.partitionBy('player_idx').orderBy('session_date_days').rangeBetween(-29,0))

sessions_silver_sec = (df_sessions_sec
                       .filter(F.col('event_date') >= F.col('first_session_date'))
            .withColumn('num_sessions_7d', 
                        F.sum(F.when(F.col("num_of_session") > 0, 1).otherwise(0)).over(window_7d))
            .withColumn('churn_7d', F.when(F.col('num_sessions_7d')==0, True).otherwise(False))
            .withColumn('num_sessions_30d', 
                        F.sum(F.when(F.col('num_of_session') > 0, 1).otherwise(0)).over(window_30d))
           .withColumn('churn_30d', F.when(F.col('num_sessions_30d')==0, True).otherwise(False))
)

In [21]:

window_7d = (Window.partitionBy('player_idx').orderBy('session_date_days').rangeBetween(1,6))

target = (sessions_silver_sec
            .withColumn('next_7d_churn', 
            (F.max(F.col("churn_7d").cast("int")).over(window_7d) == 1))
)

In [24]:
target.filter(F.col('player_idx')==84
).filter(F.col('event_date')>'2024-05-20').show(25)

+----------+----------+-----------------+------------------+-----------------+--------------+-----------------+---------------+--------+----------------+---------+-------------+
|player_idx|event_date|registration_date|first_session_date|last_session_date|num_of_session|session_date_days|num_sessions_7d|churn_7d|num_sessions_30d|churn_30d|next_7d_churn|
+----------+----------+-----------------+------------------+-----------------+--------------+-----------------+---------------+--------+----------------+---------+-------------+
|        84|2024-05-21|       2024-02-05|        2024-02-06|       2024-06-30|             1|            19864|              4|   false|              12|    false|        false|
|        84|2024-05-22|       2024-02-05|        2024-02-06|       2024-06-30|          NULL|            19865|              3|   false|              12|    false|        false|
|        84|2024-05-23|       2024-02-05|        2024-02-06|       2024-06-30|          NULL|            19866

In [25]:
sessions_silver.filter(F.col('player_idx')==87).orderBy('session_date').show()

+----------+-------+--------------------+---------+----------------+----------------+-----------+------------+-------------+-----------------+----------+---------+
|session_id|game_id|session_duration_sec|bet_count|total_bet_amount|total_win_amount|device_type|session_date|signed_amount|balance_after_txn|player_idx|days_diff|
+----------+-------+--------------------+---------+----------------+----------------+-----------+------------+-------------+-----------------+----------+---------+
+----------+-------+--------------------+---------+----------------+----------------+-----------+------------+-------------+-----------------+----------+---------+



In [26]:
sessions_silver_sec.filter(F.col('churn_7d')==True).filter(F.col('event_date')>'2024-01-07') .orderBy('event_date').show()

+----------+----------+-----------------+------------------+-----------------+--------------+-----------------+---------------+--------+----------------+---------+
|player_idx|event_date|registration_date|first_session_date|last_session_date|num_of_session|session_date_days|num_sessions_7d|churn_7d|num_sessions_30d|churn_30d|
+----------+----------+-----------------+------------------+-----------------+--------------+-----------------+---------------+--------+----------------+---------+
|     85760|2024-01-08|       2024-01-01|        2024-01-01|       2024-06-30|          NULL|            19730|              0|    true|               1|    false|
|     99961|2024-01-08|       2024-01-01|        2024-01-01|       2024-06-30|          NULL|            19730|              0|    true|               1|    false|
|     79903|2024-01-08|       2024-01-01|        2024-01-01|       2024-06-28|          NULL|            19730|              0|    true|               1|    false|
|     39735|2024

In [ ]:
df_sessions.orderBy('player_idx',).filter(F.col('event_date')=='2024-04-26').show(30, truncate=False)

In [ ]:
df_num_of_sessions.orderBy('player_idx', 'session_date').show()